# §03 Graph Algorithms — Exercises

> *"An algorithm must be seen to be believed."* — Donald Knuth

Eight graded exercises covering graph traversal through max-flow.

| Format | Description |
|--------|-------------|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code scaffold — fill in the blanks |
| **Solution** | Reference solution with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
|-------|-----------|-------|
| ★ | 1–3 | Core traversal and shortest paths |
| ★★ | 4–6 | MST, DAG DP, SCC |
| ★★★ | 7–8 | Max-flow, GPU graph algorithms |

### Topic Map

| Topic | Exercise |
|-------|----------|
| BFS shortest paths | 1 |
| DFS and cycle detection | 2 |
| Dijkstra and Bellman-Ford | 3 |
| Kruskal MST with Union-Find | 4 |
| Topological sort and DAG DP | 5 |
| Tarjan SCC | 6 |
| Max-flow / min-cut | 7 |
| GNN receptive fields & graph algorithms | 8 |


In [ ]:
import numpy as np
import heapq
from collections import deque

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [9, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

print('Setup complete.')


---

## Exercise 1 ★ — BFS Shortest Paths

**Difficulty:** ★ Foundational

Consider the undirected graph with adjacency list:
$0:[1,2]$, $1:[0,3,4]$, $2:[0,4]$, $3:[1,5]$, $4:[1,2,5]$, $5:[3,4]$.

### Part (a)
Implement BFS from source $s=0$. Return the distance array $d[\cdot]$ and parent array.

### Part (b)
Find all shortest paths from node 0 to node 5 (there may be more than one).
Implement `all_shortest_paths(adj, s, t)` using BFS.

### Part (c)
Verify the **BFS shortest-hop theorem**: $d[v]$ equals the hop distance $\delta(0,v)$
for all $v$. Count and print all distinct shortest paths from 0 to 5.

### Part (d)
Implement BFS-based **bipartiteness check**: colour nodes with 2 colours; if a conflict
arises, the graph is not bipartite. Is this graph bipartite?


In [ ]:
# Exercise 1: Your Solution

adj1 = {0:[1,2], 1:[0,3,4], 2:[0,4], 3:[1,5], 4:[1,2,5], 5:[3,4]}
n1 = 6

# (a) BFS
def my_bfs(adj, s, n):
    # YOUR CODE HERE
    d = None
    parent = None
    return d, parent

# (b) All shortest paths
def all_shortest_paths(adj, s, t, n):
    # YOUR CODE HERE
    return []

# (d) Bipartiteness check
def is_bipartite(adj, n):
    # YOUR CODE HERE
    return None, None  # (bool, colouring)

d1, par1 = my_bfs(adj1, 0, n1)
paths = all_shortest_paths(adj1, 0, 5, n1)
bip, colours = is_bipartite(adj1, n1)
print(f'd = {d1}')
print(f'Paths 0→5: {paths}')
print(f'Bipartite: {bip}')


In [ ]:
# Exercise 1: Solution

adj1 = {0:[1,2], 1:[0,3,4], 2:[0,4], 3:[1,5], 4:[1,2,5], 5:[3,4]}
n1 = 6

# (a) BFS
def my_bfs(adj, s, n):
    d = [float('inf')] * n
    parent = [-1] * n
    d[s] = 0
    queue = deque([s])
    while queue:
        u = queue.popleft()
        for v in adj.get(u, []):
            if d[v] == float('inf'):
                d[v] = d[u] + 1
                parent[v] = u
                queue.append(v)
    return d, parent

# (b) All shortest paths (BFS + backtrack)
def all_shortest_paths(adj, s, t, n):
    d, _ = my_bfs(adj, s, n)
    if d[t] == float('inf'):
        return []
    # BFS to collect all predecessors at each level
    preds = {v: [] for v in range(n)}
    for u in range(n):
        for v in adj.get(u, []):
            if d[v] == d[u] + 1:
                preds[v].append(u)
    # Backtrack
    paths = []
    def backtrack(v, path):
        if v == s:
            paths.append(list(reversed(path + [s])))
            return
        for u in preds[v]:
            backtrack(u, path + [v])
    backtrack(t, [])
    return paths

# (d) Bipartite check
def is_bipartite(adj, n):
    colour = [-1] * n
    for start in range(n):
        if colour[start] != -1:
            continue
        colour[start] = 0
        queue = deque([start])
        while queue:
            u = queue.popleft()
            for v in adj.get(u, []):
                if colour[v] == -1:
                    colour[v] = 1 - colour[u]
                    queue.append(v)
                elif colour[v] == colour[u]:
                    return False, colour
    return True, colour

d1, par1 = my_bfs(adj1, 0, n1)
paths1 = all_shortest_paths(adj1, 0, 5, n1)
bip1, col1 = is_bipartite(adj1, n1)

header('Exercise 1: BFS Shortest Paths')
print(f'd = {d1}')
check_close('d[5] = 2', [d1[5]], [2])
check_close('d vector', d1, [0,1,1,2,2,3])
print(f'Shortest paths 0→5: {paths1}  ({len(paths1)} paths)')
check_true('at least 2 shortest paths exist', len(paths1) >= 2)
check_true('all paths have length d[5]+1', all(len(p)==d1[5]+1 for p in paths1))
print(f'Bipartite: {bip1}, colours: {col1}')
check_true('graph is not bipartite (odd cycle present)', not bip1)
print('\nTakeaway: BFS gives exact shortest-hop distances in O(n+m).')
print('Multiple shortest paths = multiple predecessor choices at some BFS layer.')


---

## Exercise 2 ★ — DFS, Edge Classification, and Cycle Detection

**Difficulty:** ★ Foundational

Consider the directed graph with edges:
$0\to1$, $0\to2$, $1\to3$, $2\to3$, $3\to4$, $4\to2$ (creates a cycle), $1\to4$.

### Part (a)
Implement DFS with discovery and finish timestamps. Return `disc[]`, `fin[]`, `parent[]`.

### Part (b)
Classify every edge as tree, back, forward, or cross.
Identify all **back edges** — they indicate directed cycles.

### Part (c)
Identify the cycle in the graph from the back edge(s).
Reconstruct the cycle by following parent pointers from the back edge's head to its tail.

### Part (d)
Remove the back edge(s). Verify the resulting graph is a DAG by confirming no back edges
exist in a fresh DFS. What is a valid topological ordering of the resulting DAG?


In [ ]:
# Exercise 2: Your Solution

adj2 = {0:[1,2], 1:[3,4], 2:[3], 3:[4], 4:[2]}
n2 = 5

def my_dfs_classify(adj, n):
    # YOUR CODE HERE
    # Return disc, fin, parent, edge_types dict
    return None, None, None, {}

disc2, fin2, par2, edges2 = my_dfs_classify(adj2, n2)
print(f'disc = {disc2}')
print(f'fin  = {fin2}')
back_edges = [(u,v) for (u,v),t in edges2.items() if t=='BACK']
print(f'Back edges: {back_edges}')


In [ ]:
# Exercise 2: Solution

adj2 = {0:[1,2], 1:[3,4], 2:[3], 3:[4], 4:[2]}
n2 = 5

def my_dfs_classify(adj, n):
    colour = ['WHITE'] * n
    disc   = [-1] * n
    fin    = [-1] * n
    parent = [-1] * n
    time   = [0]
    edge_types = {}

    def visit(u):
        colour[u] = 'GREY'
        disc[u] = time[0]; time[0] += 1
        for v in adj.get(u, []):
            if colour[v] == 'WHITE':
                edge_types[(u,v)] = 'TREE'
                parent[v] = u
                visit(v)
            elif colour[v] == 'GREY':
                edge_types[(u,v)] = 'BACK'
            elif colour[v] == 'BLACK':
                edge_types[(u,v)] = 'FORWARD' if disc[v]>disc[u] else 'CROSS'
        colour[u] = 'BLACK'
        fin[u] = time[0]; time[0] += 1

    for v in range(n):
        if colour[v] == 'WHITE': visit(v)
    return disc, fin, parent, edge_types

disc2, fin2, par2, edges2 = my_dfs_classify(adj2, n2)

def reconstruct_cycle(back_edge, parent):
    """Reconstruct cycle from back edge (head→tail via parent pointers)."""
    tail, head = back_edge  # tail→head is the back edge
    # head is ancestor of tail; follow parents from tail to head
    cycle = [tail]
    cur = parent[tail]
    while cur != head and cur != -1:
        cycle.append(cur)
        cur = parent[cur]
    cycle.append(head)
    return list(reversed(cycle))

header('Exercise 2: DFS Edge Classification')
print('Timestamps:')
for v in range(n2):
    print(f'  node {v}: disc={disc2[v]}, fin={fin2[v]}, parent={par2[v]}')
print('Edge classifications:')
for (u,v),t in sorted(edges2.items()):
    print(f'  ({u},{v}): {t}')

back_edges2 = [(u,v) for (u,v),t in edges2.items() if t=='BACK']
print(f'Back edges: {back_edges2}')
check_true('back edge (4,2) found', (4,2) in back_edges2)

if back_edges2:
    cycle = reconstruct_cycle(back_edges2[0], par2)
    print(f'Cycle: {cycle}')

# (d) Remove back edges and verify DAG
adj2_dag = {u: [v for v in vs if (u,v) not in back_edges2] for u, vs in adj2.items()}
_, _, _, edges2_dag = my_dfs_classify(adj2_dag, n2)
back_dag = [(u,v) for (u,v),t in edges2_dag.items() if t=='BACK']
check_true('no back edges after removal (is DAG)', len(back_dag) == 0)
print('\nTakeaway: Back edges in DFS ↔ directed cycles. Remove them → DAG.')


---

## Exercise 3 ★ — Dijkstra vs Bellman-Ford

**Difficulty:** ★ Foundational

Graph with nodes $\{s,a,b,c,d\}$ and directed edges:
$(s,a,10)$, $(s,b,3)$, $(b,a,4)$, $(b,c,8)$, $(a,c,1)$, $(c,d,2)$, $(a,d,7)$.

### Part (a)
Run Dijkstra from $s$. Show the priority queue extraction order and distances.

### Part (b)
Run Bellman-Ford from $s$. Show the distance array after each round.
Verify both algorithms give the same final distances.

### Part (c)
Add edge $(b,a,-6)$ (negative weight). Dijkstra will give wrong results;
Bellman-Ford will give correct results. Demonstrate and compare.

### Part (d)
Add a negative cycle: edge $(d,b,-20)$. Run Bellman-Ford and detect the cycle.


In [ ]:
# Exercise 3: Your Solution

edges3 = [('s','a',10),('s','b',3),('b','a',4),('b','c',8),('a','c',1),('c','d',2),('a','d',7)]
nodes3 = ['s','a','b','c','d']
n3 = 5
# Convert to integer indices
idx = {n:i for i,n in enumerate(nodes3)}
adj3 = {idx[u]:[] for u in nodes3}
edg3 = []
for u,v,w in edges3:
    adj3[idx[u]].append((idx[v],w))
    edg3.append((idx[u],idx[v],w))

def my_dijkstra(adj, s, n):
    # YOUR CODE HERE
    return [None]*n

def my_bellman_ford(edges, s, n):
    # YOUR CODE HERE — return (d, has_neg_cycle)
    return [None]*n, False

d_dijk3 = my_dijkstra(adj3, 0, n3)
d_bf3, neg3 = my_bellman_ford(edg3, 0, n3)
print('Dijkstra:    ', d_dijk3)
print('Bellman-Ford:', d_bf3)


In [ ]:
# Exercise 3: Solution

edges3 = [('s','a',10),('s','b',3),('b','a',4),('b','c',8),('a','c',1),('c','d',2),('a','d',7)]
nodes3 = ['s','a','b','c','d']
n3 = 5
idx3 = {n:i for i,n in enumerate(nodes3)}

def make_graph(edges, nodes):
    idx = {n:i for i,n in enumerate(nodes)}
    n = len(nodes)
    adj = {i:[] for i in range(n)}
    edg = []
    for u,v,w in edges:
        adj[idx[u]].append((idx[v],w))
        edg.append((idx[u],idx[v],w))
    return adj, edg, n, idx

def my_dijkstra(adj, s, n):
    d = [float('inf')] * n; d[s] = 0
    heap = [(0, s)]; visited = [False]*n
    while heap:
        du, u = heapq.heappop(heap)
        if visited[u]: continue
        visited[u] = True
        for v,w in adj.get(u,[]):
            if d[u]+w < d[v]:
                d[v] = d[u]+w; heapq.heappush(heap,(d[v],v))
    return d

def my_bellman_ford(edges, s, n):
    d = [float('inf')]*n; d[s]=0
    for _ in range(n-1):
        for u,v,w in edges:
            if d[u]+w < d[v]: d[v]=d[u]+w
    neg = any(d[u]+w < d[v] for u,v,w in edges if d[u]!=float('inf'))
    return d, neg

adj3, edg3, n3, idx3 = make_graph(edges3, nodes3)
d_dijk3 = my_dijkstra(adj3, 0, n3)
d_bf3, neg3 = my_bellman_ford(edg3, 0, n3)

header('Exercise 3: Dijkstra vs Bellman-Ford')
print('Distances from s:')
for name, d_d, d_b in zip(nodes3, d_dijk3, d_bf3):
    print(f'  {name}: Dijkstra={d_d}, Bellman-Ford={d_b}')
check_true('Dijkstra == Bellman-Ford (no neg weights)', d_dijk3 == d_bf3)

# (c) Add negative edge b→a=-6
edges3c = edges3 + [('b','a',-6)]
adj3c, edg3c, n3c, _ = make_graph(edges3c, nodes3)
d_dijk3c = my_dijkstra(adj3c, 0, n3c)
d_bf3c, neg3c = my_bellman_ford(edg3c, 0, n3c)
print(f'\nWith b→a=-6:')
print(f'  Dijkstra (WRONG): {dict(zip(nodes3, d_dijk3c))}')
print(f'  Bellman-Ford (CORRECT): {dict(zip(nodes3, d_bf3c))}')
check_true('Dijkstra incorrect for a (should be 3+(-6)=-3 but Dijkstra gives >-3)',
           d_dijk3c[1] != d_bf3c[1])

# (d) Add negative cycle d→b=-20 (makes s→b→c→d→b→c→d→... a neg cycle)
edges3d = edges3c + [('d','b',-20)]
_, edg3d, _, _ = make_graph(edges3d, nodes3)
d_bf3d, neg3d = my_bellman_ford(edg3d, 0, n3)
print(f'\nWith negative cycle (d→b=-20):')
print(f'  Negative cycle detected: {neg3d}')
check_true('negative cycle detected', neg3d)
print('\nTakeaway: Dijkstra fails for negative weights; Bellman-Ford handles them')
print('and detects negative cycles via the (n)-th round test.')


---

## Exercise 4 ★★ — Kruskal's MST with Union-Find

**Difficulty:** ★★ Intermediate

Graph with nodes $\{A,B,C,D,E,F\}$ and edges:
$(A,B,7)$, $(A,D,5)$, $(B,C,8)$, $(B,D,9)$, $(B,E,7)$, $(C,E,5)$, $(D,E,15)$,
$(D,F,6)$, $(E,F,8)$, $(E,G,9)$, $(F,G,11)$.

### Part (a)
Implement Union-Find with path compression and union by rank.
Verify: `find(2)` after `union(0,1)`, `union(1,2)` returns the same root.

### Part (b)
Implement Kruskal's algorithm using your Union-Find.
Print the sorted edge order and which edges are added/skipped.

### Part (c)
Verify the **cut property**: for the cut $S=\{A,D,F\}$, $T=\{B,C,E,G\}$,
find the minimum-weight edge crossing the cut and confirm it is in the MST.

### Part (d)
Compute the MST total weight. Is the MST unique? (Check if any two edges have
equal weight that could be swapped.)


In [ ]:
# Exercise 4: Your Solution

nodes4 = ['A','B','C','D','E','F','G']
n4 = 7
idx4 = {n:i for i,n in enumerate(nodes4)}
edges4 = [('A','B',7),('A','D',5),('B','C',8),('B','D',9),('B','E',7),
           ('C','E',5),('D','E',15),('D','F',6),('E','F',8),('E','G',9),('F','G',11)]
edges4_idx = [(idx4[u],idx4[v],w) for u,v,w in edges4]

class MyUnionFind:
    def __init__(self, n):
        # YOUR CODE HERE
        pass
    def find(self, x):
        # YOUR CODE HERE (with path compression)
        return x
    def union(self, x, y):
        # YOUR CODE HERE (with union by rank)
        return False

def my_kruskal(n, edges):
    # YOUR CODE HERE
    return [], 0

mst4, w4 = my_kruskal(n4, edges4_idx)
print(f'MST edges: {[(nodes4[u],nodes4[v],w) for u,v,w in mst4]}')
print(f'MST weight: {w4}')


In [ ]:
# Exercise 4: Solution

nodes4 = ['A','B','C','D','E','F','G']
n4 = 7
idx4 = {n:i for i,n in enumerate(nodes4)}
edges4 = [('A','B',7),('A','D',5),('B','C',8),('B','D',9),('B','E',7),
           ('C','E',5),('D','E',15),('D','F',6),('E','F',8),('E','G',9),('F','G',11)]
edges4_idx = [(idx4[u],idx4[v],w) for u,v,w in edges4]

class MyUnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py: return False
        if self.rank[px] < self.rank[py]: px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]: self.rank[px] += 1
        return True

def my_kruskal(n, edges, names=None):
    sorted_edges = sorted(edges, key=lambda e: e[2])
    uf = MyUnionFind(n)
    mst = []
    print('Kruskal trace:')
    for u,v,w in sorted_edges:
        nu = names[u] if names else u
        nv = names[v] if names else v
        if uf.union(u,v):
            mst.append((u,v,w))
            print(f'  ADD  ({nu},{nv}) w={w}')
        else:
            print(f'  SKIP ({nu},{nv}) w={w} — cycle')
        if len(mst) == n-1: break
    return mst, sum(e[2] for e in mst)

header('Exercise 4: Kruskal MST')
mst4, w4 = my_kruskal(n4, edges4_idx, nodes4)
print(f'\nMST edges: {[(nodes4[u],nodes4[v],w) for u,v,w in mst4]}')
print(f'MST weight: {w4}')
check_true('MST has n-1 = 6 edges', len(mst4) == n4-1)

# (c) Cut property: S={A,D,F}, T={B,C,E,G}
S_cut = {idx4['A'], idx4['D'], idx4['F']}
T_cut = {idx4['B'], idx4['C'], idx4['E'], idx4['G']}
cut_edges = [(u,v,w) for u,v,w in edges4_idx
             if (u in S_cut and v in T_cut) or (v in S_cut and u in T_cut)]
min_cut_edge = min(cut_edges, key=lambda e:e[2])
mst_set = {(min(u,v),max(u,v)) for u,v,_ in mst4}
mce_key = (min(min_cut_edge[0],min_cut_edge[1]), max(min_cut_edge[0],min_cut_edge[1]))
print(f'\nCut S={{A,D,F}}, T={{B,C,E,G}}')
print(f'Min cut edge: ({nodes4[min_cut_edge[0]]},{nodes4[min_cut_edge[1]]}) w={min_cut_edge[2]}')
check_true('min cut edge is in MST (cut property)', mce_key in mst_set)
print('\nTakeaway: Cut property guarantees Kruskal adds the correct edge at each step.')


---

## Exercise 5 ★★ — Topological Sort and DAG Shortest/Longest Paths

**Difficulty:** ★★ Intermediate

Consider a project scheduling DAG with tasks $\{0,1,2,3,4,5\}$ and
precedence edges (task must complete before successor can start):
$(0,1,3)$, $(0,2,5)$, $(1,3,2)$, $(1,4,4)$, $(2,3,1)$, $(2,5,6)$,
$(3,5,3)$, $(4,5,2)$. Edge weight = task duration.

### Part (a)
Implement Kahn's topological sort. Verify the graph is a DAG (no cycle detected).
Print a valid topological ordering of the tasks.

### Part (b)
Implement DAG shortest path (earliest start time) using the topological order.
Node 0 starts at time 0. $d[v] = $ earliest time $v$ can start.

### Part (c)
Implement DAG longest path (critical path). The critical path determines
the minimum project completion time.
Print the critical path and its length.

### Part (d)
Verify: removing any edge on the critical path increases the minimum completion
time. Show this for at least one critical edge.


In [ ]:
# Exercise 5: Your Solution

from collections import deque

# (task_from, task_to, duration)
edges5 = [(0,1,3),(0,2,5),(1,3,2),(1,4,4),(2,3,1),(2,5,6),(3,5,3),(4,5,2)]
n5 = 6
adj5 = {i:[] for i in range(n5)}
for u,v,w in edges5:
    adj5[u].append((v,w))

def my_kahn(adj, n):
    # YOUR CODE HERE — return topo order or None if cycle
    return list(range(n))

def dag_sssp(adj, topo_order, s, n):
    # YOUR CODE HERE — earliest start times
    return [0]*n

def dag_longest(adj, topo_order, s, n):
    # YOUR CODE HERE — latest completion (longest path)
    return [0]*n, [-1]*n

topo5 = my_kahn(adj5, n5)
earliest = dag_sssp(adj5, topo5, 0, n5)
latest, crit_par = dag_longest(adj5, topo5, 0, n5)
print(f'Topo order: {topo5}')
print(f'Earliest start: {earliest}')
print(f'Latest (longest path): {latest}')


In [ ]:
# Exercise 5: Solution

from collections import deque

edges5 = [(0,1,3),(0,2,5),(1,3,2),(1,4,4),(2,3,1),(2,5,6),(3,5,3),(4,5,2)]
n5 = 6
adj5 = {i:[] for i in range(n5)}
for u,v,w in edges5:
    adj5[u].append((v,w))

def my_kahn(adj, n):
    indeg = [0]*n
    for u in range(n):
        for v,_ in adj.get(u,[]):
            indeg[v] += 1
    q = deque(v for v in range(n) if indeg[v]==0)
    order = []
    while q:
        u = q.popleft(); order.append(u)
        for v,_ in adj.get(u,[]):
            indeg[v] -= 1
            if indeg[v]==0: q.append(v)
    return order if len(order)==n else None

def dag_sssp(adj, topo, s, n):
    d = [float('inf')]*n; d[s]=0
    for u in topo:
        if d[u]==float('inf'): continue
        for v,w in adj.get(u,[]):
            if d[u]+w < d[v]: d[v]=d[u]+w
    return d

def dag_longest(adj, topo, s, n):
    d = [-float('inf')]*n; d[s]=0
    par = [-1]*n
    for u in topo:
        if d[u]==-float('inf'): continue
        for v,w in adj.get(u,[]):
            if d[u]+w > d[v]:
                d[v]=d[u]+w; par[v]=u
    return d, par

topo5 = my_kahn(adj5, n5)
earliest5 = dag_sssp(adj5, topo5, 0, n5)
latest5, cpar5 = dag_longest(adj5, topo5, 0, n5)

# Reconstruct critical path
def reconstruct_dag_path(par, t):
    path=[]; cur=t
    while cur!=-1: path.append(cur); cur=par[cur]
    return list(reversed(path))

sink = n5-1
crit_path = reconstruct_dag_path(cpar5, sink)

header = lambda t: print('\n'+'='*len(t)+'\n'+t+'\n'+'='*len(t))
header('Exercise 5: Topological Sort & DAG DP')
print(f'Topological order: {topo5}')
check_true('is valid topological order', topo5 is not None)
print(f'Earliest start times: {earliest5}')
print(f'Longest path (critical): {latest5}')
print(f'Critical path: {crit_path}')
print(f'Project duration: {latest5[sink]}')
check_close('project duration = 11', [latest5[sink]], [11])

# (d) Remove one critical edge and verify longer completion
crit_edges = [(crit_path[i],crit_path[i+1]) for i in range(len(crit_path)-1)]
u_rem, v_rem = crit_edges[0]
adj5_mod = {u: [(vv,ww) for vv,ww in vs if not (u==u_rem and vv==v_rem)]
            for u,vs in adj5.items()}
topo5_mod = my_kahn(adj5_mod, n5)
lat5_mod, _ = dag_longest(adj5_mod, topo5_mod, 0, n5)
print(f'\nRemove critical edge ({u_rem},{v_rem}):')
print(f'  New project duration: {lat5_mod[sink]}')
check_true('removing critical edge does not decrease duration', lat5_mod[sink] >= latest5[sink])
print('\nTakeaway: Topological sort enables O(n+m) DAG DP. Critical path = longest path.')
print('In neural nets: topo sort = forward-pass order; critical path = gradient bottleneck.')


---

## Exercise 6 ★★ — Tarjan's Strongly Connected Components

**Difficulty:** ★★ Intermediate

Consider the directed graph with edges:
$0\to1$, $1\to2$, $2\to0$ (cycle), $1\to3$, $3\to4$, $4\to5$, $5\to3$ (cycle),
$2\to6$, $6\to7$.

### Part (a)
Implement Tarjan's algorithm. Trace `disc[]` and `low[]` values during the DFS.
Identify all SCCs.

### Part (b)
Build the **condensation DAG**: contract each SCC to a single node and add
edges between SCCs. Verify the condensation is a DAG (no back edges in DFS).

### Part (c)
Identify the **source SCCs** (in-degree 0 in condensation) and **sink SCCs**
(out-degree 0). Which SCC is the unique source and which is the unique sink?

### Part (d)
A directed graph is **strongly connected** iff it has exactly one SCC.
Add a single edge to make the entire graph strongly connected.
Verify with Tarjan.


In [ ]:
# Exercise 6: Your Solution

adj6 = {0:[1], 1:[2,3], 2:[0,6], 3:[4], 4:[5], 5:[3], 6:[7], 7:[]}
n6 = 8

def my_tarjan(adj, n):
    # YOUR CODE HERE — return list of SCCs
    return [[v] for v in range(n)]

sccs6 = my_tarjan(adj6, n6)
print('SCCs:', [sorted(s) for s in sccs6])


In [ ]:
# Exercise 6: Solution

adj6 = {0:[1], 1:[2,3], 2:[0,6], 3:[4], 4:[5], 5:[3], 6:[7], 7:[]}
n6 = 8

def my_tarjan(adj, n):
    disc=[−1]*n; low=[0]*n; on_stack=[False]*n
    stack=[]; sccs=[]; time=[0]
    def visit(v):
        disc[v]=low[v]=time[0]; time[0]+=1
        stack.append(v); on_stack[v]=True
        for w in adj.get(v,[]):
            if disc[w]==-1:
                visit(w); low[v]=min(low[v],low[w])
            elif on_stack[w]:
                low[v]=min(low[v],disc[w])
        if low[v]==disc[v]:
            scc=[]
            while True:
                w=stack.pop(); on_stack[w]=False; scc.append(w)
                if w==v: break
            sccs.append(scc)
    for v in range(n):
        if disc[v]==-1: visit(v)
    return sccs

# Fix: Python doesn't allow − as unary before ] in list literal; use -1
def my_tarjan(adj, n):
    disc=[-1]*n; low=[0]*n; on_stack=[False]*n
    stack=[]; sccs=[]; time=[0]
    def visit(v):
        disc[v]=low[v]=time[0]; time[0]+=1
        stack.append(v); on_stack[v]=True
        for w in adj.get(v,[]):
            if disc[w]==-1:
                visit(w); low[v]=min(low[v],low[w])
            elif on_stack[w]:
                low[v]=min(low[v],disc[w])
        if low[v]==disc[v]:
            scc=[]
            while True:
                w=stack.pop(); on_stack[w]=False; scc.append(w)
                if w==v: break
            sccs.append(scc)
    for v in range(n):
        if disc[v]==-1: visit(v)
    return sccs

sccs6 = my_tarjan(adj6, n6)
scc_sets6 = [frozenset(s) for s in sccs6]
expected6 = [frozenset({0,1,2}),frozenset({3,4,5}),frozenset({6}),frozenset({7})]

header = lambda t: print('\n'+'='*len(t)+'\n'+t+'\n'+'='*len(t))
header('Exercise 6: Tarjan SCC')
print('SCCs found:', [sorted(s) for s in sccs6])
check_true('correct 4 SCCs found', set(scc_sets6)==set(expected6))

# (b) Condensation DAG
scc_id = {}
for i, scc in enumerate(sccs6):
    for v in scc: scc_id[v] = i

n_cond = len(sccs6)
cond_edges = set()
for u in range(n6):
    for v in adj6.get(u,[]):
        su, sv = scc_id[u], scc_id[v]
        if su != sv: cond_edges.add((su,sv))

print(f'\nCondensation edges: {sorted(cond_edges)}')

# (c) Source and sink SCCs
in_deg  = {i:0 for i in range(n_cond)}
out_deg = {i:0 for i in range(n_cond)}
for u,v in cond_edges: in_deg[v]+=1; out_deg[u]+=1
sources = [i for i in range(n_cond) if in_deg[i]==0]
sinks   = [i for i in range(n_cond) if out_deg[i]==0]
print(f'Source SCCs (in-deg 0): {[sorted(sccs6[i]) for i in sources]}')
print(f'Sink SCCs (out-deg 0):  {[sorted(sccs6[i]) for i in sinks]}')

# (d) Make strongly connected: add edge 7→0
adj6_sc = {u: list(vs) for u,vs in adj6.items()}
adj6_sc[7].append(0)
sccs6_sc = my_tarjan(adj6_sc, n6)
check_true('one SCC after adding 7→0 (strongly connected)', len(sccs6_sc)==1)
print('\nTakeaway: SCCs reveal hierarchical graph structure. Condensation is always a DAG.')
print('Adding one edge from deepest sink to root source makes the graph strongly connected.')


---

## Exercise 7 ★★★ — Maximum Flow and Minimum Cut

**Difficulty:** ★★★ Advanced

Flow network: source $s=0$, sink $t=5$, nodes $\{s,a,b,c,d,t\}$.
Edge capacities: $(s,a,10)$, $(s,b,8)$, $(a,c,7)$, $(a,d,5)$,
$(b,c,6)$, $(b,d,4)$, $(c,t,9)$, $(d,t,6)$.

### Part (a)
Implement Edmonds-Karp (Ford-Fulkerson with BFS augmenting paths).
Print each augmenting path and its bottleneck capacity.

### Part (b)
Find the maximum flow value. Verify using the Max-Flow Min-Cut theorem:
find an $s$-$t$ cut with capacity equal to the max flow.

### Part (c)
Draw the residual graph after the algorithm terminates.
Which edges are saturated (at full capacity)? Which have residual capacity?

### Part (d)
Model this as a data routing problem: $s$ = data centre, $a,b$ = regional hubs,
$c,d$ = distribution nodes, $t$ = users. Capacities = bandwidth (Gbps).
What is the bottleneck link? How would upgrading which single edge most increase throughput?


In [ ]:
# Exercise 7: Your Solution

from collections import deque

n7 = 6  # s=0,a=1,b=2,c=3,d=4,t=5
names7 = ['s','a','b','c','d','t']
cap7_init = [[0]*n7 for _ in range(n7)]
for u,v,c in [(0,1,10),(0,2,8),(1,3,7),(1,4,5),(2,3,6),(2,4,4),(3,5,9),(4,5,6)]:
    cap7_init[u][v] = c

def my_edmonds_karp(n, cap_init):
    # YOUR CODE HERE — return (max_flow, steps, min_cut_S)
    return 0, [], [0]

import copy
cap7 = copy.deepcopy(cap7_init)
mf7, steps7, S7 = my_edmonds_karp(n7, cap7)
print(f'Max flow: {mf7}')
print(f'Min cut S: {[names7[v] for v in S7]}')


In [ ]:
# Exercise 7: Solution

import copy
from collections import deque

n7 = 6
names7 = ['s','a','b','c','d','t']
cap7_init = [[0]*n7 for _ in range(n7)]
for u,v,c in [(0,1,10),(0,2,8),(1,3,7),(1,4,5),(2,3,6),(2,4,4),(3,5,9),(4,5,6)]:
    cap7_init[u][v] = c

def my_edmonds_karp(n, cap_orig):
    cap = copy.deepcopy(cap_orig)
    s, t = 0, n-1
    total = 0; steps = []
    while True:
        # BFS for shortest augmenting path
        par = [-1]*n; par[s]=s
        q = deque([s])
        while q and par[t]==-1:
            u=q.popleft()
            for v in range(n):
                if par[v]==-1 and cap[u][v]>0:
                    par[v]=u; q.append(v)
        if par[t]==-1: break
        # Bottleneck
        delta=float('inf'); v=t; path=[]
        while v!=s:
            u=par[v]; delta=min(delta,cap[u][v]); path.append((u,v)); v=u
        path.reverse()
        # Augment
        v=t
        while v!=s:
            u=par[v]; cap[u][v]-=delta; cap[v][u]+=delta; v=u
        total+=delta; steps.append((path,delta))
    # Min cut: BFS reachable in residual
    reachable=[False]*n; reachable[s]=True
    q=deque([s])
    while q:
        u=q.popleft()
        for v in range(n):
            if not reachable[v] and cap[u][v]>0:
                reachable[v]=True; q.append(v)
    S=[v for v in range(n) if reachable[v]]
    return total, steps, S, cap

mf7, steps7, S7, residual7 = my_edmonds_karp(n7, cap7_init)
T7 = [v for v in range(n7) if v not in S7]

header = lambda t: print('\n'+'='*len(t)+'\n'+t+'\n'+'='*len(t))
header('Exercise 7: Max-Flow / Min-Cut')
print('Augmenting paths:')
for i,(path,delta) in enumerate(steps7):
    pstr='→'.join(names7[u] for u,_ in path)+f'→{names7[path[-1][1]]}'
    print(f'  Step {i+1}: {pstr}  bottleneck={delta}')
print(f'\nMax flow = {mf7}')
check_true('max flow = 15', mf7 == 15)

# Min-cut capacity
cut_cap = sum(cap7_init[u][v] for u in S7 for v in T7)
print(f'Min cut: S={[names7[v] for v in S7]}, T={[names7[v] for v in T7]}')
print(f'Cut capacity = {cut_cap}')
check_true('max flow = min cut capacity', mf7 == cut_cap)

# (c) Saturated edges
print('\nSaturated edges (residual cap = 0):')
for u in range(n7):
    for v in range(n7):
        if cap7_init[u][v] > 0 and residual7[u][v] == 0:
            print(f'  ({names7[u]},{names7[v]}) cap={cap7_init[u][v]} — SATURATED')

# (d) Bottleneck upgrade analysis
print('\nBottleneck analysis (upgrade which edge gains most?):')
best_gain=0; best_edge=None
for u in range(n7):
    for v in range(n7):
        if cap7_init[u][v]>0:
            cap_mod=copy.deepcopy(cap7_init)
            cap_mod[u][v]+=5  # +5 Gbps upgrade
            mf_mod,_,_,_ = my_edmonds_karp(n7,cap_mod)
            gain=mf_mod-mf7
            if gain>best_gain: best_gain=gain; best_edge=(u,v)
if best_edge:
    print(f'  Best upgrade: ({names7[best_edge[0]]},{names7[best_edge[1]]}) gains {best_gain} Gbps')
else:
    print('  No single edge upgrade improves max flow (all min-cut edges saturated).')
print('\nTakeaway: Max-Flow = Min-Cut. Saturated min-cut edges are the bottlenecks.')
print('Upgrading a non-bottleneck edge never improves throughput.')


---

## Exercise 8 ★★★ — GNN Receptive Fields and Graph Algorithm Connections

**Difficulty:** ★★★ Advanced

This exercise makes the BFS ↔ GNN connection mathematically precise and
demonstrates the Bellman-Ford ↔ RL value iteration duality.

### Part (a)
Build a random Erdős–Rényi graph $G(20, 0.2)$ (20 nodes, each edge included
with probability 0.2). For each node $v$, compute the BFS $k$-hop neighbourhood
$\Gamma^k(v)$ for $k = 1, 2, 3$. Plot the neighbourhood size distribution.

### Part (b)
Simulate a 2-layer GCN (no learned weights — just the normalised aggregation
$\hat{A}^2 H^{(0)}$). For two nodes $u, v$ with identical 2-hop BFS neighbourhoods,
verify they produce identical GCN embeddings (WL indistinguishability).

### Part (c)
Implement **value iteration** (RL Bellman equation) on a small MDP graph:
- States = nodes in a 6-node graph
- Actions = move to adjacent node
- Reward = $+10$ for reaching node 5 (goal), $-1$ per step otherwise
- Discount $\gamma = 0.9$

Show that value iteration converges to the same answer as Dijkstra (shortest
path = highest value = fewest steps to goal).

### Part (d)
Measure empirical running time of BFS, Dijkstra, and Bellman-Ford on
Erdős–Rényi graphs $G(n, 5/n)$ for $n \in \{100, 500, 1000, 2000\}$.
Fit lines on a log-log plot to estimate empirical scaling exponents.


In [ ]:
# Exercise 8: Your Solution

import numpy as np
from collections import deque

# (a) Generate G(n,p) and compute k-hop neighbourhoods
def erdos_renyi(n, p, seed=42):
    rng = np.random.default_rng(seed)
    adj = {i:[] for i in range(n)}
    for u in range(n):
        for v in range(u+1,n):
            if rng.random()<p:
                adj[u].append(v); adj[v].append(u)
    return adj

n8, p8 = 20, 0.2
adj8 = erdos_renyi(n8, p8)

# YOUR CODE HERE for (a)-(d)
print('Graph built with', sum(len(v) for v in adj8.values())//2, 'edges')


In [ ]:
# Exercise 8: Solution

import numpy as np, time
from collections import deque
import heapq

def erdos_renyi(n, p, seed=42):
    rng=np.random.default_rng(seed)
    adj={i:[] for i in range(n)}
    for u in range(n):
        for v in range(u+1,n):
            if rng.random()<p:
                adj[u].append(v); adj[v].append(u)
    return adj

def bfs_dist(adj,s,n):
    d=[float('inf')]*n; d[s]=0; q=deque([s])
    while q:
        u=q.popleft()
        for v in adj.get(u,[]):
            if d[v]==float('inf'): d[v]=d[u]+1; q.append(v)
    return d

# (a) k-hop neighbourhood sizes
n8,p8=20,0.2
adj8=erdos_renyi(n8,p8)
m8=sum(len(v) for v in adj8.values())//2

header = lambda t: print('\n'+'='*len(t)+'\n'+t+'\n'+'='*len(t))
header('Exercise 8: GNN Receptive Fields')
print(f'G(20,0.2): {m8} edges, avg degree {2*m8/n8:.1f}')
for k in [1,2,3]:
    sizes=[sum(1 for d in bfs_dist(adj8,v,n8) if d<=k and d>=0) for v in range(n8)]
    print(f'  k={k}: avg neighbourhood size = {np.mean(sizes):.1f}, max = {max(sizes)}')

# (b) GCN WL indistinguishability
A8=np.zeros((n8,n8))
for u,vs in adj8.items():
    for v in vs: A8[u,v]=1.0
At=A8+np.eye(n8)
d_deg=At.sum(1)
Dinv=np.diag(1/np.sqrt(d_deg))
Ahat=Dinv@At@Dinv
Ahat2=Ahat@Ahat
H0=np.eye(n8)  # identity features: each node knows its own ID
H2=Ahat2@H0
print('\n(b) Nodes with identical 2-hop GCN rows (WL-indistinguishable):')
found=False
for u in range(n8):
    for v in range(u+1,n8):
        if np.allclose(H2[u],H2[v],atol=1e-6):
            print(f'  Nodes {u} and {v} have identical 2-layer GCN embeddings')
            found=True
if not found:
    print('  No WL-indistinguishable pair found (identity features distinguish all nodes here)')
    print('  WL indistinguishability typically occurs with symmetric graph structure')

# (c) Value iteration = Bellman-Ford on MDP
adj_mdp={0:[1,2],1:[0,3],2:[0,3],3:[1,2,4],4:[3,5],5:[]}
n_mdp=6; goal=5; gamma=0.9
V=np.zeros(n_mdp)
for _ in range(200):
    V_new=np.zeros(n_mdp)
    for s in range(n_mdp):
        if s==goal: V_new[s]=10.0; continue
        if not adj_mdp[s]: continue
        V_new[s]=max(-1+gamma*V[a] for a in adj_mdp[s])
    if np.allclose(V,V_new,atol=1e-6): break
    V=V_new
print('\n(c) Value iteration (MDP):')
print(f'  V = {V.round(3)}')
print(f'  Highest value node (closest to goal): {int(np.argmax(V[:goal]))}')

# BFS hop distance to goal (for comparison)
d_goal=bfs_dist(adj_mdp,goal,n_mdp)
print(f'  BFS dist to goal: {d_goal}')
check_true('value decreases with hop distance to goal',
    all(V[u]>=V[v] or d_goal[u]>=d_goal[v] for u in range(goal) for v in range(goal)))

# (d) Empirical scaling
def make_rand(n,seed=0):
    rng=np.random.default_rng(seed)
    adj={i:[] for i in range(n)}
    edg=[]
    for u in range(n):
        for _ in range(5):  # avg degree 5
            v=int(rng.integers(0,n))
            if v!=u: adj[u].append(v); edg.append((u,v,1.0))
    return adj,edg

print('\n(d) Empirical scaling:')
print(f"{'n':>6} | {'BFS(ms)':>9} | {'Dijkstra(ms)':>13} | {'B-Ford(ms)':>12}")
print('-'*48)
for n_sc in [100,500,1000,2000]:
    adj_sc,edg_sc=make_rand(n_sc)
    adj_w_sc={u:[(v,w) for v,w in [(vv,1.0) for vv in vs]] for u,vs in adj_sc.items()}
    t0=time.perf_counter(); bfs_dist(adj_sc,0,n_sc); t_bfs=(time.perf_counter()-t0)*1e3
    t0=time.perf_counter()
    d_dijk=[float('inf')]*n_sc; d_dijk[0]=0
    hp=[(0,0)]; vis=[False]*n_sc
    while hp:
        du,u=heapq.heappop(hp)
        if vis[u]: continue
        vis[u]=True
        for v in adj_sc.get(u,[]):
            if du+1<d_dijk[v]: d_dijk[v]=du+1; heapq.heappush(hp,(d_dijk[v],v))
    t_dijk=(time.perf_counter()-t0)*1e3
    t0=time.perf_counter()
    d_bf=[float('inf')]*n_sc; d_bf[0]=0
    for _ in range(min(n_sc-1,10)):  # only 10 rounds to keep tractable
        for u,v,w in edg_sc:
            if d_bf[u]+w<d_bf[v]: d_bf[v]=d_bf[u]+w
    t_bf=(time.perf_counter()-t0)*1e3
    print(f'{n_sc:>6} | {t_bfs:>9.2f} | {t_dijk:>13.2f} | {t_bf:>12.2f}')

print('\nTakeaway: BFS = O(n+m) is fastest. Dijkstra adds log factor from heap.')
print('Bellman-Ford O(nm) is slowest but handles negative weights and detects negative cycles.')
print('In GNNs: the message-passing step is BFS aggregation with learned weights.')


---

## What to Review After Finishing

- [ ] **BFS correctness** — can you prove $d[v] = \delta(s,v)$ by induction on hop distance?
- [ ] **Dijkstra invariant** — can you explain why greedy extraction is correct for non-negative weights?
- [ ] **Bellman-Ford ↔ Bellman equation** — can you write the RL recurrence and identify it as Bellman-Ford?
- [ ] **Kruskal cut property** — can you state and use the cut property to prove Kruskal is correct?
- [ ] **Tarjan low-link** — can you explain why `low[v] = disc[v]` identifies SCC roots?
- [ ] **Max-flow min-cut duality** — can you state the theorem and describe the min-cut construction?
- [ ] **GNN receptive field** — can you explain why a $k$-layer GNN sees exactly the BFS $k$-hop neighbourhood?

## References

1. Cormen et al. — *Introduction to Algorithms* (CLRS), Ch. 22–26
2. Kleinberg & Tardos — *Algorithm Design*, Ch. 3–4, 7
3. Kipf & Welling (2017) — *Semi-Supervised Classification with Graph Convolutional Networks*
4. Zhu et al. (2021) — *Neural Bellman-Ford Networks* (NBFNet)
5. Boykov & Kolmogorov (2004) — *An Experimental Comparison of Min-Cut/Max-Flow Algorithms*
